# 📈 Notebook 2 — Baseline Models (v3: AutoSearch 17 Profiles)
**Datathon VinTelligence 2026 | Revenue & COGS Forecasting**

**Philosophy:** Prophet as backbone → LightGBM corrects residuals.

**Requires:** Run `01_data_preparation.ipynb` first.

**Outputs saved to** `data/data_clean/`:
- `v3_submission.csv` — best v3 forecast
- `v3_cv_results.csv` — cross-validation MAE for all 17 profiles

## 0. Setup & Imports

In [33]:
# Install dependencies if needed
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

try:
    import prophet
except ImportError:
    install('prophet')

try:
    import lightgbm
except ImportError:
    install('lightgbm')

In [34]:
import warnings
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats as scipy_stats
from prophet import Prophet
import lightgbm as lgb
from sklearn.linear_model import LinearRegression

warnings.filterwarnings('ignore')
np.random.seed(42)
import os, sys

def _detect_base_dir():
    """Auto-detect BASE_DIR: works on Colab, local Windows, and local Linux."""
    # 1. Google Colab
    if 'google.colab' in sys.modules or os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        return Path('/content/drive/My Drive/Datathon_VinTelligence')
    # 2. Local — find project root (the folder containing 'data/')
    try:
        nb_dir = Path(__file__).resolve().parent
    except NameError:
        nb_dir = Path().resolve()  # in .ipynb, cwd = notebook folder
    for candidate in [nb_dir, nb_dir.parent, nb_dir.parent.parent]:
        if (candidate / 'data').exists():
            return candidate
    return nb_dir.parent  # last-resort fallback

BASE_DIR  = _detect_base_dir()
DATA_DIR  = BASE_DIR / 'data' / 'datathon-2026-round-1'
CLEAN_DIR = BASE_DIR / 'data' / 'data_clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'ENV       : {"Colab" if os.path.exists("/content") else "Local"}')
print(f'BASE_DIR  : {BASE_DIR}')
print(f'DATA_DIR  : {DATA_DIR}')
print(f'CLEAN_DIR : {CLEAN_DIR}')

OUT_DIR   = OUTPUT_DIR
FORECAST_START = pd.Timestamp('2023-01-01')
FORECAST_END   = pd.Timestamp('2024-07-01')
TRAIN_END      = pd.Timestamp('2022-12-31')
N_FORECAST     = 548

TARGET_COLS = ['Revenue', 'COGS']
print('Setup complete.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ENV       : Colab
BASE_DIR  : /content/drive/My Drive/Datathon_VinTelligence
DATA_DIR  : /content/drive/My Drive/Datathon_VinTelligence/data/datathon-2026-round-1
CLEAN_DIR : /content/drive/My Drive/Datathon_VinTelligence/data/data_clean
Setup complete.


## 1. Load Processed Data

In [35]:
df_sales = pd.read_csv(CLEAN_DIR / 'sales_clean.csv', parse_dates=['Date'])
df_sales = df_sales.sort_values('Date').reset_index(drop=True)

df_cal   = pd.read_csv(CLEAN_DIR / 'calendar_features.csv', parse_dates=['ds'], index_col='ds')
df_holidays = pd.read_csv(CLEAN_DIR / 'prophet_holidays.csv', parse_dates=['ds'])

# Aux profiles
web_profile   = pd.read_csv(CLEAN_DIR / 'aux_web_profile.csv')
order_profile = pd.read_csv(CLEAN_DIR / 'aux_orders_profile.csv')
return_profile= pd.read_csv(CLEAN_DIR / 'aux_returns_profile.csv')
promo_profile = pd.read_csv(CLEAN_DIR / 'aux_promos_profile.csv')

# Historical medians
hist_medians = {}
for col in ['revenue', 'cogs']:
    for grp in ['doy', 'dow', 'month']:
        key = f'{grp}_{col}'
        s = pd.read_csv(CLEAN_DIR / f'hist_median_{key}.csv', index_col=0)
        hist_medians[key] = s

# Sample submission
df_sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', parse_dates=['Date'])

print(f'Sales: {len(df_sales):,} rows | {df_sales.Date.min().date()} → {df_sales.Date.max().date()}')
print(f'Calendar: {df_cal.shape}')

Sales: 3,833 rows | 2012-07-04 → 2022-12-31
Calendar: (4381, 46)


## 2. Seasonal Naive Baselines (3 Variants)

In [36]:
def weighted_median(values, weights):
    """Compute weighted median."""
    sorted_idx = np.argsort(values)
    vals_sorted = np.array(values)[sorted_idx]
    wts_sorted  = np.array(weights)[sorted_idx]
    cum_wts = np.cumsum(wts_sorted)
    cutoff  = cum_wts[-1] / 2.0
    return vals_sorted[np.searchsorted(cum_wts, cutoff)]


def compute_naive_blend(df_sales: pd.DataFrame, forecast_dates: pd.DatetimeIndex,
                        col: str) -> pd.Series:
    """
    3-variant Seasonal Naive blend:
      V1: Naive364 — DOW-preserving, fallback offsets
      V2: SeasonalWindowMedian — same DOW, ±7 days DOY
      V3: Trend-adjusted blend
    """
    history = df_sales.set_index('Date')[col]
    years   = df_sales.Date.dt.year.unique()

    # YoY growth adjustment
    rev_by_year = df_sales.groupby(df_sales.Date.dt.year)[col].sum()
    if len(rev_by_year) >= 2:
        g = rev_by_year.iloc[-1] / rev_by_year.iloc[-2]
        growth = np.clip(g, 0.85, 1.20)
    else:
        growth = 1.0

    naive364_vals = []
    window_med_vals = []

    for dt in forecast_dates:
        # ── Variant 1: Naive364 ─────────────────────────────────────────
        v364 = np.nan
        for offset in [364, 371, 357, 728]:
            candidate = dt - pd.Timedelta(days=offset)
            if candidate in history.index:
                v364 = history[candidate]
                break
        if np.isnan(v364):
            v364 = history.iloc[-1]  # last known

        # ── Variant 2: Seasonal Window Median ───────────────────────────
        target_dow = dt.dayofweek
        target_doy = dt.dayofyear
        hist_with_meta = df_sales[df_sales.Date.dt.year < dt.year].copy()
        if len(hist_with_meta) == 0:
            hist_with_meta = df_sales.copy()
        hist_with_meta = hist_with_meta.assign(
            hdow = hist_with_meta.Date.dt.dayofweek,
            hdoy = hist_with_meta.Date.dt.dayofyear,
        )
        # Circular DOY distance
        doy_dist = hist_with_meta.hdoy.apply(
            lambda d: min(abs(d - target_doy), 365 - abs(d - target_doy))
        )
        mask = (hist_with_meta.hdow == target_dow) & (doy_dist <= 7)
        if mask.sum() > 0:
            vwm = hist_with_meta[mask][col].median()
        else:
            vwm = v364

        naive364_vals.append(v364)
        window_med_vals.append(vwm)

    # Blend: 50/50 → trend-adjust
    blend = 0.5 * np.array(naive364_vals) + 0.5 * np.array(window_med_vals)
    blend_adjusted = blend * growth

    return pd.Series(blend_adjusted, index=forecast_dates, name=f'naive_{col.lower()}')


# Build forecast dates
forecast_dates = pd.date_range(FORECAST_START, FORECAST_END, freq='D')
print(f'Forecast dates: {len(forecast_dates)} days')

# Compute naive for both targets
naive_rev  = compute_naive_blend(df_sales, forecast_dates, 'Revenue')
naive_cogs = compute_naive_blend(df_sales, forecast_dates, 'COGS')

print(f'Naive Revenue  mean: {naive_rev.mean():,.0f}')
print(f'Naive COGS     mean: {naive_cogs.mean():,.0f}')
print(f'Sample sub Rev mean: {df_sample.set_index("Date").Revenue.mean():,.0f}')

Forecast dates: 548 days
Naive Revenue  mean: 4,362,829
Naive COGS     mean: 3,613,991
Sample sub Rev mean: 3,249,795


## 3. Prophet Model (Feature Extractor)

In [37]:
def fit_prophet(df_sales: pd.DataFrame, col: str,
                forecast_dates: pd.DatetimeIndex,
                df_holidays: pd.DataFrame,
                changepoint_prior_scale: float = 0.05,
                seasonality_mode: str = 'multiplicative',
                yearly_fourier: int = 20,
                train_start_year: int = None) -> tuple:
    """
    Fit Prophet and return (train_components_df, forecast_components_df).
    Components: yhat, trend, weekly, yearly, holidays, monthly.
    """
    df = df_sales[['Date', col]].rename(columns={'Date': 'ds', col: 'y'})
    if train_start_year:
        df = df[df.ds.dt.year >= train_start_year]

    # Cap/floor for logistic growth
    cap   = df.y.quantile(0.995) * 1.25
    floor = max(df.y.quantile(0.005) * 0.75, 0)
    df['cap']   = cap
    df['floor'] = floor

    model = Prophet(
        growth='logistic',
        yearly_seasonality=yearly_fourier,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode=seasonality_mode,
        changepoint_prior_scale=changepoint_prior_scale,
        changepoint_range=0.9,
        holidays=df_holidays,
        holidays_prior_scale=10.0,
    )
    model.add_seasonality('monthly',   period=30.5,  fourier_order=5)
    model.add_seasonality('quarterly', period=91.25, fourier_order=3)
    model.fit(df)

    # Predict on training dates (for LightGBM feature)
    train_future = model.make_future_dataframe(periods=0, freq='D')
    train_future['cap']   = cap
    train_future['floor'] = floor
    train_pred = model.predict(train_future)

    # Predict on forecast dates
    fc_df = pd.DataFrame({'ds': forecast_dates, 'cap': cap, 'floor': floor})
    fc_pred = model.predict(fc_df)

    component_cols = ['ds', 'yhat', 'trend', 'weekly', 'yearly', 'holidays',
                      'additive_terms', 'multiplicative_terms']
    # Add monthly/quarterly if present
    for extra in ['monthly', 'quarterly']:
        if extra in train_pred.columns:
            component_cols.append(extra)

    train_comp = train_pred[[c for c in component_cols if c in train_pred.columns]]
    fc_comp    = fc_pred[[c for c in component_cols if c in fc_pred.columns]]

    return train_comp, fc_comp, model


print('Prophet fitting functions ready.')

Prophet fitting functions ready.


## 4. Feature Matrix Builder for LightGBM

In [38]:
def build_feature_matrix(dates: pd.DatetimeIndex,
                         df_cal: pd.DataFrame,
                         prophet_comp: pd.DataFrame,
                         hist_medians: dict,
                         web_profile: pd.DataFrame,
                         order_profile: pd.DataFrame,
                         return_profile: pd.DataFrame,
                         promo_profile: pd.DataFrame,
                         col_suffix: str,
                         train_end: pd.Timestamp,
                         n_total: int) -> pd.DataFrame:
    """
    Build 60+ feature matrix for LightGBM.
    """
    # Calendar features
    feat = df_cal.loc[dates].copy()
    feat.index.name = 'ds'

    # Horizon features
    all_fc_dates = pd.date_range('2023-01-01', periods=n_total, freq='D')
    horizon_map = {d: i for i, d in enumerate(all_fc_dates)}
    feat['forecast_horizon'] = feat.index.map(lambda d: horizon_map.get(d, 0))
    feat['days_since_start'] = (feat.index - pd.Timestamp('2012-07-04')).days
    # Linear trend
    t_vals = feat['days_since_start'].values.reshape(-1, 1)
    feat['linear_trend'] = feat['days_since_start']  # raw for now

    # Prophet components
    pc = prophet_comp.set_index('ds')
    for comp in ['yhat', 'trend', 'weekly', 'yearly', 'holidays', 'monthly', 'quarterly']:
        if comp in pc.columns:
            feat[f'prophet_{comp}'] = pc[comp].reindex(dates).values

    # Historical medians
    col = col_suffix.lower()
    # Fix: Use .squeeze() to ensure 1D mapping for .map()
    doy_map = hist_medians.get(f'doy_{col}', pd.Series()).squeeze()
    dow_map = hist_medians.get(f'dow_{col}', pd.Series()).squeeze()
    month_map = hist_medians.get(f'month_{col}', pd.Series()).squeeze()

    feat['hist_doy_target']   = feat['doy'].map(doy_map)
    feat['hist_dow_target']   = feat['dow'].map(dow_map)
    feat['hist_month_target'] = feat['month'].map(month_map)

    # Aux: web profile
    feat_with_meta = feat.reset_index()
    feat_with_meta = feat_with_meta.merge(web_profile, on=['month', 'dow'], how='left')
    feat_with_meta = feat_with_meta.merge(order_profile, on=['month', 'dow'], how='left')
    feat_with_meta = feat_with_meta.merge(return_profile, on='month', how='left')
    feat_with_meta = feat_with_meta.merge(promo_profile, on='month', how='left')
    feat_with_meta = feat_with_meta.set_index('ds')

    return feat_with_meta

## 5. LightGBM Residual Model

In [39]:
def fit_lgbm_residual(train_df: pd.DataFrame,
                      X_train: pd.DataFrame,
                      prophet_train: pd.DataFrame,
                      col: str,
                      n_estimators: int = 1200,
                      lgb_objective: str = 'mae') -> lgb.LGBMRegressor:
    """
    LightGBM model to correct Prophet residuals.
    Target: actual - prophet_yhat
    """
    pc = prophet_train.set_index('ds')
    train_dates = train_df.Date
    actuals     = train_df.set_index('Date')[col]

    prophet_yhat = pc['yhat'].reindex(train_dates.values).values
    residuals = actuals.values - prophet_yhat

    # Recency sample weights
    n = len(train_df)
    ranks = np.arange(1, n+1)
    sample_weight = 1.0 + 2.0 * (ranks / n) ** 1.5

    # Feature selection: drop non-numeric / id cols
    drop_cols = ['date', 'year']  # already captured as features
    X = X_train.drop(columns=[c for c in drop_cols if c in X_train.columns], errors='ignore')
    X = X.select_dtypes(include=[np.number]).fillna(X.median())

    model = lgb.LGBMRegressor(
        n_estimators=n_estimators,
        learning_rate=0.02,
        num_leaves=31,
        min_child_samples=30,
        reg_alpha=0.5,
        reg_lambda=5.0,
        objective=lgb_objective,
        subsample=0.85,
        colsample_bytree=0.85,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X.values, residuals, sample_weight=sample_weight)
    return model, X.columns.tolist()


print('LightGBM residual trainer ready.')

LightGBM residual trainer ready.


## 6. Rolling Year Cross-Validation

In [40]:
def rolling_year_cv(df_sales: pd.DataFrame, df_cal: pd.DataFrame,
                    df_holidays: pd.DataFrame, hist_medians: dict,
                    web_profile, order_profile, return_profile, promo_profile,
                    col: str,
                    config: dict) -> float:
    """
    Rolling year cross-validation: folds = [2020, 2021, 2022].
    fold_weights = [1.0, 1.5, 2.0] (recent matters more).
    Returns weighted avg MAE.
    """
    fold_years   = [2020, 2021, 2022]
    fold_weights = np.array([1.0, 1.5, 2.0])
    fold_weights = fold_weights / fold_weights.sum()

    maes = []
    for fold_year in fold_years:
        train_end_cv = pd.Timestamp(f'{fold_year-1}-12-31')
        val_start_cv = pd.Timestamp(f'{fold_year}-01-01')
        val_end_cv   = pd.Timestamp(f'{fold_year}-12-31')

        train_cv = df_sales[df_sales.Date <= train_end_cv].copy()
        val_cv   = df_sales[(df_sales.Date >= val_start_cv) & (df_sales.Date <= val_end_cv)]

        if len(train_cv) < 365 or len(val_cv) == 0:
            continue

        val_dates = val_cv.Date.values

        # Naive
        naive_fc = compute_naive_blend(train_cv, pd.DatetimeIndex(val_dates), col)

        # Prophet
        try:
            prophet_train, prophet_fc, _ = fit_prophet(
                train_cv, col, pd.DatetimeIndex(val_dates), df_holidays,
                changepoint_prior_scale=config.get('cps', 0.05),
                seasonality_mode=config.get('seasonality_mode', 'multiplicative'),
                yearly_fourier=config.get('yearly_fourier', 20),
                train_start_year=config.get('train_start_year', None),
            )
        except Exception as e:
            print(f'  Prophet failed fold {fold_year}: {e}')
            continue

        prophet_pred = prophet_fc.set_index('ds')['yhat'].reindex(val_dates).values

        # LightGBM
        try:
            X_train_cv = build_feature_matrix(
                train_cv.Date.values, df_cal, prophet_train,
                hist_medians, web_profile, order_profile, return_profile, promo_profile,
                col, train_end_cv, 365
            )
            X_val_cv = build_feature_matrix(
                val_dates, df_cal, prophet_train,
                hist_medians, web_profile, order_profile, return_profile, promo_profile,
                col, train_end_cv, 365
            )
            lgbm_model, feat_cols = fit_lgbm_residual(
                train_cv, X_train_cv, prophet_train, col,
                n_estimators=config.get('n_estimators', 1000),
                lgb_objective=config.get('lgb_objective', 'mae'),
            )
            X_val_np = X_val_cv[feat_cols].fillna(X_val_cv[feat_cols].median()).values
            lgbm_correction = lgbm_model.predict(X_val_np)
            hybrid_pred = prophet_pred + lgbm_correction
        except Exception as e:
            print(f'  LGBM failed fold {fold_year}: {e}')
            hybrid_pred = prophet_pred

        # Grid-search best blend weights (fast version)
        actuals_val = val_cv[col].values
        best_mae = np.inf
        best_pred = hybrid_pred
        for wn in np.arange(0.0, 1.05, 0.1):
            for wh in np.arange(0.0, 1.05 - wn, 0.1):
                wp = 1.0 - wn - wh
                if wp < 0:
                    continue
                pred = wn * naive_fc.values + wp * prophet_pred + wh * hybrid_pred
                mae = np.mean(np.abs(actuals_val - pred))
                if mae < best_mae:
                    best_mae = mae
                    best_pred = pred

        maes.append(best_mae)

    if not maes:
        return np.inf
    # Pad if some folds failed
    while len(maes) < len(fold_years):
        maes.append(maes[-1])
    return float(np.dot(maes, fold_weights[:len(maes)]))


print('Rolling CV function ready.')

Rolling CV function ready.


## 7. AutoSearch — 17 Search Profiles

In [41]:
# Define the 17 search profiles
SEARCH_PROFILES = [
    # Profile 1: Default
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 2: Low CPS
    {'cps': 0.03, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 3: High CPS
    {'cps': 0.10, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 4: Additive mode
    {'cps': 0.05, 'seasonality_mode': 'additive', 'yearly_fourier': 15,
     'lgb_objective': 'mae', 'n_estimators': 1000, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 5: More Fourier
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 25,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 6: Fewer Fourier
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 15,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 7: Huber loss
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'huber', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 8: More estimators
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1500, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 9: From 2016
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': 2016, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 10: From 2018
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': 2018, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 11: High decay
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.6},
    # Profile 12: No dynamic blend
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': False, 'decay_strength': 0.5},
    # Profile 13: Low CPS + high Fourier
    {'cps': 0.03, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 25,
     'lgb_objective': 'mae', 'n_estimators': 1000, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 14: High CPS + additive
    {'cps': 0.10, 'seasonality_mode': 'additive', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 15: From 2016 + huber
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'huber', 'n_estimators': 1200, 'train_start_year': 2016, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 16: 800 estimators
    {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 800, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5},
    # Profile 17: Low CPS + from 2016 + high decay
    {'cps': 0.03, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20,
     'lgb_objective': 'mae', 'n_estimators': 1000, 'train_start_year': 2016, 'dynamic_blend': True, 'decay_strength': 0.6},
]

print(f'Total search profiles: {len(SEARCH_PROFILES)}')

Total search profiles: 17


In [42]:
# ⚠️ AutoSearch is compute-intensive. Run a quick version (3 profiles) for speed,
#    or set RUN_FULL_SEARCH=True to run all 17 profiles.
RUN_FULL_SEARCH = False  # Set True for competition submission

profiles_to_run = SEARCH_PROFILES if RUN_FULL_SEARCH else SEARCH_PROFILES[:3]

cv_results = []
for i, cfg in enumerate(profiles_to_run):
    print(f'\n[{i+1}/{len(profiles_to_run)}] Running profile: {cfg}')
    for col in TARGET_COLS:
        mae = rolling_year_cv(
            df_sales, df_cal, df_holidays, hist_medians,
            web_profile, order_profile, return_profile, promo_profile,
            col, cfg
        )
        cv_results.append({'profile_id': i+1, 'col': col, 'cv_mae': mae, **cfg})
        print(f'  {col} CV MAE: {mae:,.0f}')

cv_df = pd.DataFrame(cv_results)
cv_df.to_csv(OUT_DIR / 'v3_cv_results.csv', index=False)
print('\nCV Results:')
display(cv_df[['profile_id', 'col', 'cv_mae', 'cps', 'seasonality_mode', 'n_estimators']].pivot(
    index='profile_id', columns='col', values='cv_mae'
).round(0))


[1/3] Running profile: {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20, 'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005900 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6420
[LightGBM] [Info] Number of data points in the train set: 2737, number of used features: 62
[LightGBM] [Info] Start training from score 4012.632080
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.076916 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6425
[LightGBM] [Info] Number of data points in the train set: 3103, number of used features: 63
[LightGBM] [Info] Start training from score 9383.599609
[LightGBM] [Info] Auto-choosing 

col,COGS,Revenue
profile_id,,
1,637763.0,722857.0
2,650825.0,732790.0
3,643923.0,725174.0


## 8. Full Forecast with Best Profile

In [43]:
def dynamic_blend_weights(n: int, wn: float, wp: float, wh: float,
                          decay_strength: float = 0.5) -> tuple:
    """Dynamic horizon-dependent blending weights."""
    t = np.linspace(0, 1, n)
    wh_t = wh * (1 - decay_strength * t)       # LGBM weight decreases
    wp_t = wp * (1 + 0.4 * t)                  # Prophet weight increases
    wn_t = wn * (1 + 0.15 * t)                 # Naive weight increases slightly
    # Normalize
    total = wh_t + wp_t + wn_t
    return wh_t / total, wp_t / total, wn_t / total


def generate_v3_forecast(df_sales: pd.DataFrame, df_cal: pd.DataFrame,
                         df_holidays: pd.DataFrame, hist_medians: dict,
                         web_profile, order_profile, return_profile, promo_profile,
                         forecast_dates: pd.DatetimeIndex,
                         best_config: dict) -> dict:
    """Generate final v3 forecast using best config."""
    results = {}

    for col in TARGET_COLS:
        print(f'\nFitting {col}...')

        # 1. Naive
        naive_fc = compute_naive_blend(df_sales, forecast_dates, col)

        # 2. Prophet
        prophet_train, prophet_fc, prophet_model = fit_prophet(
            df_sales, col, forecast_dates, df_holidays,
            changepoint_prior_scale=best_config.get('cps', 0.05),
            seasonality_mode=best_config.get('seasonality_mode', 'multiplicative'),
            yearly_fourier=best_config.get('yearly_fourier', 20),
            train_start_year=best_config.get('train_start_year', None),
        )
        prophet_pred = prophet_fc.set_index('ds')['yhat'].reindex(forecast_dates).values

        # 3. LightGBM residuals
        train_dates = df_sales.Date.values
        X_train = build_feature_matrix(
            train_dates, df_cal, prophet_train, hist_medians,
            web_profile, order_profile, return_profile, promo_profile,
            col, TRAIN_END, N_FORECAST
        )
        X_fc = build_feature_matrix(
            forecast_dates.values, df_cal, prophet_train, hist_medians,
            web_profile, order_profile, return_profile, promo_profile,
            col, TRAIN_END, N_FORECAST
        )
        lgbm_model, feat_cols = fit_lgbm_residual(
            df_sales, X_train, prophet_train, col,
            n_estimators=best_config.get('n_estimators', 1200),
            lgb_objective=best_config.get('lgb_objective', 'mae'),
        )
        X_fc_np = X_fc[feat_cols].fillna(X_fc[feat_cols].median()).values
        lgbm_correction = lgbm_model.predict(X_fc_np)
        hybrid_pred = prophet_pred + lgbm_correction

        # 4. Grid-search blend weights using 2022 as final holdout check
        best_wn, best_wp, best_wh = 0.3, 0.3, 0.4
        if cv_df is not None and len(cv_df) > 0:
            # Just use default weights from plan
            pass

        # 5. Dynamic blending
        n = len(forecast_dates)
        if best_config.get('dynamic_blend', True):
            wh_t, wp_t, wn_t = dynamic_blend_weights(
                n, best_wn, best_wp, best_wh,
                best_config.get('decay_strength', 0.5)
            )
            final_pred = wn_t * naive_fc.values + wp_t * prophet_pred + wh_t * hybrid_pred
        else:
            final_pred = best_wn * naive_fc.values + best_wp * prophet_pred + best_wh * hybrid_pred

        final_pred = np.maximum(final_pred, 0)  # clip to non-negative
        results[col] = final_pred
        print(f'  {col} forecast mean: {final_pred.mean():,.0f}')

    return results


# Select best config
if len(cv_df) > 0:
    # Average MAE across Revenue and COGS per profile
    avg_mae = cv_df.groupby('profile_id')['cv_mae'].mean()
    best_profile_id = avg_mae.idxmin()
    best_config_v3 = SEARCH_PROFILES[best_profile_id - 1]
    print(f'Best profile: #{best_profile_id} with avg MAE={avg_mae.min():,.0f}')
    print(f'Config: {best_config_v3}')
else:
    best_config_v3 = SEARCH_PROFILES[0]  # fallback to default

# Fix the `build_feature_matrix` function to handle `hist_medians` correctly
def build_feature_matrix(dates: pd.DatetimeIndex,
                         df_cal: pd.DataFrame,
                         prophet_comp: pd.DataFrame,
                         hist_medians: dict,
                         web_profile: pd.DataFrame,
                         order_profile: pd.DataFrame,
                         return_profile: pd.DataFrame,
                         promo_profile: pd.DataFrame,
                         col_suffix: str,
                         train_end: pd.Timestamp,
                         n_total: int) -> pd.DataFrame:
    """
    Build 60+ feature matrix for LightGBM.
    """
    # Calendar features
    feat = df_cal.loc[dates].copy()
    feat.index.name = 'ds'

    # Horizon features
    all_fc_dates = pd.date_range('2023-01-01', periods=n_total, freq='D')
    horizon_map = {d: i for i, d in enumerate(all_fc_dates)}
    feat['forecast_horizon'] = feat.index.map(lambda d: horizon_map.get(d, 0))
    feat['days_since_start'] = (feat.index - pd.Timestamp('2012-07-04')).days
    # Linear trend
    t_vals = feat['days_since_start'].values.reshape(-1, 1)
    feat['linear_trend'] = feat['days_since_start']  # raw for now

    # Prophet components
    pc = prophet_comp.set_index('ds')
    for comp in ['yhat', 'trend', 'weekly', 'yearly', 'holidays', 'monthly', 'quarterly']:
        if comp in pc.columns:
            feat[f'prophet_{comp}'] = pc[comp].reindex(dates).values

    # Historical medians
    col = col_suffix.lower()
    # Ensure the mapping object is a Series (1-dimensional) using .squeeze()
    doy_map = hist_medians.get(f'doy_{col}', pd.Series()).squeeze()
    dow_map = hist_medians.get(f'dow_{col}', pd.Series()).squeeze()
    month_map = hist_medians.get(f'month_{col}', pd.Series()).squeeze()

    feat['hist_doy_target']   = feat['doy'].map(doy_map)
    feat['hist_dow_target']   = feat['dow'].map(dow_map)
    feat['hist_month_target'] = feat['month'].map(month_map)

    # Aux: web profile
    feat_with_meta = feat.reset_index()
    feat_with_meta = feat_with_meta.merge(web_profile, on=['month', 'dow'], how='left')
    feat_with_meta = feat_with_meta.merge(order_profile, on=['month', 'dow'], how='left')
    feat_with_meta = feat_with_meta.merge(return_profile, on='month', how='left')
    feat_with_meta = feat_with_meta.merge(promo_profile, on='month', how='left')
    feat_with_meta = feat_with_meta.set_index('ds')

    return feat_with_meta

# Generate forecast
v3_results = generate_v3_forecast(
    df_sales, df_cal, df_holidays, hist_medians,
    web_profile, order_profile, return_profile, promo_profile,
    forecast_dates, best_config_v3
)
print('\nv3 forecast generated.')

Best profile: #1 with avg MAE=680,310
Config: {'cps': 0.05, 'seasonality_mode': 'multiplicative', 'yearly_fourier': 20, 'lgb_objective': 'mae', 'n_estimators': 1200, 'train_start_year': None, 'dynamic_blend': True, 'decay_strength': 0.5}

Fitting Revenue...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002686 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6432
[LightGBM] [Info] Number of data points in the train set: 3833, number of used features: 63
[LightGBM] [Info] Start training from score -5599.727539
  Revenue forecast mean: 3,593,408

Fitting COGS...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002568 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6431
[LightGBM] [Info] Number of data points in the train set: 3833, number of used features: 63
[LightGBM] [Info] Start training from score 10898.388672


## 9. Save v3 Submission

In [46]:
sub_v3 = pd.DataFrame({
    'Date': forecast_dates,
    'Revenue': v3_results['Revenue'],
    'COGS':    v3_results['COGS'],
})
sub_v3['Date'] = sub_v3['Date'].dt.strftime('%Y-%m-%d')
sub_v3.to_csv(OUT_DIR / 'v3_submission.csv', index=False)

# Validation check
assert len(sub_v3) == N_FORECAST, f'Expected {N_FORECAST} rows, got {len(sub_v3)}'
assert (sub_v3.Revenue > 0).all(), 'Some Revenue predictions are non-positive'
assert (sub_v3.COGS > 0).all(), 'Some COGS predictions are non-positive'

print(f'v3 submission saved: {OUT_DIR}/v3_submission.csv')
print(f'Rows: {len(sub_v3)}')
print(f'Revenue: mean={sub_v3.Revenue.mean():,.0f}, min={sub_v3.Revenue.min():,.0f}, max={sub_v3.Revenue.max():,.0f}')
print(f'COGS:    mean={sub_v3.COGS.mean():,.0f}, min={sub_v3.COGS.min():,.0f}, max={sub_v3.COGS.max():,.0f}')

v3 submission saved: /content/drive/My Drive/Datathon_VinTelligence/output/v3_submission.csv
Rows: 548
Revenue: mean=3,593,408, min=855,705, max=7,530,457
COGS:    mean=3,104,173, min=540,129, max=6,662,387
